In [4]:
import math

In [5]:
def class_of_bolt(grade:float)->tuple[float,float]:
    """
    Returns fub and fyb
    """
    fub=int(grade)*100
    dec=grade-int(grade)
    fyb=dec*fub
    return fub,round(fyb,1)

In [2]:
class_of_bolt(4.6)

(400, 240.0)

In [6]:
def net_area_of_bolt(d:float)->tuple[float,float]:
    """
    Returns the net area of the bolt,Anb(mm2)
    where d- diameter of the bolt,mm 
    """
    Asb=math.pi/4*(d**2)
    Anb=0.78*Asb

    return round(Anb,2),round(Asb,2)

In [25]:
net_area_of_bolt(20)

(245.04, 314.16)

In [7]:
def hole_diameter(d:float)->float:
    """
    IS 800:2007 (Clause 10.2.1), the diameter of a standard bolt hole (d₀) is larger than the nominal diameter of the bolt (d) 
    to allow easy insertion and account for minor misalignments. For standard clearance holes, the extra clearance added to the bolt 
    diameter depends on the bolt size: 1.0 mm extra for 12 to 14 mm bolts, 2.0 mm extra for 16 to 24 mm bolts, and 3.0 mm extra for bolts 
    larger than 24 mm.
    """
    if d>12 and d<14:
        do=d+1
    elif d>16 and d<24:
        do=d+2
    elif d>24:
        do=d+3

    return do

In [10]:
hole_diameter(20)

22

In [8]:
def edge_distance (d:float,t:float,fy:float)->tuple[float,float]:
    """
    Returns the maximum and minimum edge/end distance
    """
    do=hole_diameter(d)
    emin=1.5*do
    eps=(250/fy)**(1/2)
    emax=12*t*eps

    return emin,emax

In [12]:
edge_distance(20,10,250)

(33.0, 120.0)

In [9]:
def pitch(d:float,t:float,type:str)->tuple[float,float]:
    """
    Returns the minimum pitch,p_min,mm & maximum pitch,p_max,mm
    d-diameter of the bolt,mm
    t-thickness of the thinner plate,mm
    type-type of member(tension/compression)

    IS 800:2007 (Clause 10.2.2), The distance between centre of fasteners shall not be
    less than 2.5 times the nominal diameter of the fastener.
    """
    p_min=2.5*d
    if (type=='compression'):
        p_max=min(12*t,200)
    elif (type=='tension'):
        p_max=min(16*t,200)
    return p_min,p_max



In [14]:
pitch(20,10,'compression')[0]

50.0

In [10]:
def shear_strength_of_bolt(grade:float,d:float,nn:int,ns:int,lj:float,lg:float,tpk:float)->float:
    """
    returns the shear strength of the bolt
    """
    fub=class_of_bolt(grade)[0]
    Anb,Asb=net_area_of_bolt(d)
    if lj>(15*d):
     rf_lj=1.075-(lj/(200*d))
     rf_lj = max(0.75, min(rf_lj, 1.0))
    else:
     rf_lj=1
    if lg>(5*d):
     rf_lg=(8*d)/(lg+(3*d))
     rf_lg = min(rf_lg, rf_lj)
    else:
     rf_lg=1
    if tpk>6:
     rf_pk=1-(0.0125*tpk)
    else:
     rf_pk=1
    V_dsb=(fub/(math.sqrt(3)*1.25))*((nn*Anb)+(ns*Asb))*rf_lg*rf_lj*rf_pk*10**-3
    return round(V_dsb,2)

In [29]:
shear_strength_of_bolt(4.6,20,1,0,150,50,5)

45.27

In [11]:
def tensile_strength_of_bolt(grade:float,d:float)->float:
    """
    returns the tensile strength of the bolt
    """
    fub=class_of_bolt(grade)[0]
    Anb=net_area_of_bolt(d)[0]
    T_db=(0.9*fub*Anb)/1.25*10**-3
    return round(T_db,2)
    

In [7]:
tensile_strength_of_bolt(4.6,20)

70.57

In [12]:
def bearing_strength_of_bolt(d:float,fu:float,grade:float,tmin:float,t:float)->float:
    """
    returns the bearing strength of the bolt
    """
    e=edge_distance(d,tmin,fu)[0]
    p=pitch(d,tmin,'compression')[0]
    do=hole_diameter(d)
    fub=class_of_bolt(grade)[0]
    kb=min((e/(3*do)),((p/(3*do))-0.25),(fu/fub),1)
    V_dpb=(2.5*kb*fu*(d*t))/1.25*10**-3
    return round(V_dpb,2)

In [10]:
bearing_strength_of_bolt(20,410,4.6,14,28)

229.6

In [13]:
def design_strength_of_bolt(grade:float,d:float,nn:int,ns:int,lj:float,lg:float,tpk:float,
                            fu:float,tmin:float,t:float)->float:
    """
    """
    V_dsb=shear_strength_of_bolt(grade,d,nn,ns,lj,lg,tpk)
    V_dpb=bearing_strength_of_bolt(d,fu,grade,tmin,t)
    T_db=tensile_strength_of_bolt(grade,d)
    Vb=min(V_dsb,V_dpb,T_db)
    return round(Vb,2)
    
    
    
    

In [14]:
design_strength_of_bolt(4.6,20,1,0,150,50,5,410,14,28)

45.27

In [1]:
def design_strength_of_butt_weld(lw:float,tmin:float,fy:float,fu:float,weld_type:str,weld_penetration:str)->tuple[float,float]:
   """
   returns the design strength of weld
   """
   if (weld_penetration=='single'):
    te=(5/8)*tmin
    f=fu
   elif (weld_penetration=='double'):
    te=tmin
    f=fy
   if (weld_type=='shop weld'):
      psf=1.25
   elif (weld_type=='field weld'):
      psf=1.5

   T_dw=(f/psf)*(lw*te)*10**-3
   V_dw=0.57*(f/psf)*(lw*te)*10**-3

   return round(T_dw,2),round(V_dw,2)

In [3]:
design_strength_of_butt_weld(150,12,240,250,'shop weld','single')[0]

225.0

In [17]:
def angle_of_fusion(theta:float)->float:
    """
    """
    if 60<theta<90:
        K=0.7
    elif 91<theta<100:
        K=0.65
    elif 101<theta<106:
        K=0.6
    elif 107<theta<113:
        K=0.55
    elif 114<theta<120:
        K=0.5
    else:
        K=0.7
    return K
        

In [18]:
angle_of_fusion(60)

0.7

In [19]:
def design_strength_of_fillet_weld(s:float,lw:float,weld_type:str,fu:float,theta:float)->tuple[float,float]:
    """
    returns the design strength of weld
    """
    K=angle_of_fusion(theta)
    tt=K*s
    if (weld_type=='shop weld'):
          psf=1.25
    elif (weld_type=='field weld'):
          psf=1.5
    T_dw=(fu/psf)*(lw*tt)*10**-3
    V_dw=(fu/(math.sqrt(3)*psf))*(lw*tt)*10**-3
    return round(T_dw,2),round(V_dw,2)
    

In [20]:
design_strength_of_fillet_weld(6,216,'shop weld',410,60)

(297.56, 171.8)

In [31]:
import pandas as pd


In [32]:
df=pd.read_csv('steel_tables_is.csv')

In [33]:
df=df.set_index('Section')

In [34]:
rolled_steel_beam=df.copy()

In [8]:
rolled_steel_beam

,W_kg/m,W_N/m,Area,h,bf,tf,tw,Ixx,Iyy,rxx,...,r1,r2,D_deg,h1,h2,b1,C,g,g1_min,Max_flange_rivet_mm
Section,,,,,,,,,,,,,,,,,,,,,
ISJB 150,7.1,69.7,9.01,150,50,4.6,3.0,322.1,9.2,5.98,...,5.0,1.5,91.5,130.4,9.80,23.50,3.00,30,45.0,6
ISJB 175,8.1,79.5,10.28,175,50,4.8,3.2,479.3,9.7,6.83,...,5.0,1.5,91.5,155.0,10.00,23.40,3.10,30,45.0,6
ISJB 200,9.9,97.1,12.64,200,60,5.0,3.4,780.7,17.3,7.86,...,5.0,1.5,91.5,179.5,10.25,28.38,3.20,30,45.0,6
ISJB 225,12.8,125.6,16.28,225,80,5.0,3.7,1308.5,40.5,8.97,...,6.5,1.5,91.5,201.5,11.95,38.15,3.35,40,45.0,12
ISLB 75,6.1,59.8,7.71,75,50,5.0,3.7,72.7,10.0,3.07,...,6.5,2.0,91.5,51.7,11.65,23.15,3.35,30,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ISHB 350,72.4,710.2,92.21,350,250,11.6,10.1,19802.8,2510.5,14.65,...,12.0,6.0,94.0,296.0,27.00,119.95,6.55,140,60.0,32
ISHB 400,77.4,759.3,98.66,400,250,12.7,9.1,28063.5,2728.3,16.87,...,14.0,7.0,94.0,340.1,29.90,120.45,6.05,140,65.0,32
ISHB 400,82.2,806.4,104.68,400,250,12.7,10.6,28823.5,2783.0,16.61,...,14.0,7.0,94.0,340.1,29.90,119.70,6.80,140,65.0,32


In [7]:
def rolled_steel_beam(beam:str,W:float)->tuple[float,float,float,float,float]:
    """
    """
    df=pd.read_csv('rolled_steel_beams.csv')
    df=df.set_index('Section')
    rolled_steel_beam=df.copy()
    condition = ((rolled_steel_beam.index == beam) & (rolled_steel_beam["W_N/m"] == W))
    steel_beam = rolled_steel_beam.loc[condition].iloc[0]
    

    Area = steel_beam["Area"]
    h = steel_beam["h"]
    bf = steel_beam["bf"]
    tf = steel_beam["tf"]
    rzz = steel_beam["rzz"]
    ryy = steel_beam["ryy"]

    return Area, h, bf, tf, rzz, ryy
    


In [8]:
rolled_steel_beam('ISHB 350',710.20)

(92.21, 350, 250, 11.6, 14.65, 5.22)

In [63]:
def buckling_class(beam:str,W:float)->str:
    """
    Returns the buckling class of the member
    """   
    section=rolled_steel_beam(beam,W)
    h=section[1]
    bf=section[2]
    tf=section[3]
    rzz,ryy=section[4],section[5]
        
    rmin=min(ryy,rzz)

    if h/bf>1.2 and tf<=40:
        if rmin==rzz:
            buckling_class='a'
        elif rmin==ryy:
            buckling_class='b'
    elif h/bf>1.2 and 40<=tf<=100:
        if rmin==rzz:
            buckling_class='b'
        elif rmin==ryy:
            buckling_class='c'
    elif h/bf<=1.2 and tf<=100:
        if rmin==rzz:
            buckling_class='b'
        elif rmin==ryy:
            buckling_class='c'
    elif h/bf<=1.2 and tf>100:
        if rmin==rzz:
            buckling_class='d'
        elif rmin==ryy:
            buckling_class='d'

    return buckling_class


In [64]:
buckling_class("ISHB 350", 710.20)

'b'

In [65]:
def effective_length_factor(condition: str) -> float:
    """
    Returns effective length factor K
    as per IS 800:2007 Table 11.
    """

    if condition == "fixed_fixed":
        K = 0.65

    elif condition == "fixed_pinned":
        K = 0.80

    elif condition == "pinned_pinned":
        K = 1.00

    elif condition == "fixed_guided":
        K = 1.20

    elif condition == "fixed_free":
        K = 2.00

    else:
        raise ValueError("Invalid end restraint condition")

    return K

In [66]:
effective_length_factor('fixed_fixed')

0.65

In [69]:
import sections as sections

In [70]:
def design_compressive_strength(beam:str,W:float,L:float,fy:float,psf:float,condition:str)->float:
    """
    to determine the design compressive strength of the member
    """
    section=sections.rolled_steel_beam(beam,W)
    Area=section[0]
    rzz,ryy=section[4],section[5]
    buckling_class=sections.buckling_class(beam,W)
    rmin=min(rzz,ryy)*10
    K=effective_length_factor(condition)
    E=2*10**5
    
    fcc=(math.pi**2*E)/(((K*L*1000)/rmin)**2)
    lamda=math.sqrt(fy/fcc)
    buckling_class=sections.buckling_class(beam,W)
    alpha=sections.imperfection_factor(buckling_class)
    phi=0.5*(1+(alpha*(lamda-0.2))+(lamda**2))
    fcd=(fy/psf)/(phi+math.sqrt((phi**2)-(lamda**2)))
    Pcd=fcd*Area*100*10**-3

    return round(Pcd,2)

    
    
    

In [71]:
design_compressive_strength('ISHB 350',710.20,4,250,1.1,'fixed_fixed')






1794.69

In [7]:
def imperfection_factor(buckling_class:str)->float:
    """
    """
    if (buckling_class=='a'):
        alpha=0.21
    elif (buckling_class=='b'):
        alpha=0.34
    elif (buckling_class=='c'):
        alpha=0.49
    elif (buckling_class=='d'):
        alpha=0.76
        
    return alpha
    

In [87]:
imperfection_factor('b')

0.34

In [22]:
def gross_section_yielding(fy:float,Ag:float)->float:
    """
    """
    gamma_m0=1.1
    T_dg=(fy/gamma_m0)*Ag*10**-3
    return round(T_dg,2)
    

In [5]:
gross_section_yielding(250,1200)

272.73

In [23]:
def net_section_rupture_plates(fu:float,Anet:float)->float:
    """
    """
    gamma_m1=1.25
    T_dn=(0.9*fu/gamma_m1)*Anet*10**-3
    return round(T_dn,2)
    

In [7]:
net_section_rupture_plates(410,760)

224.35

In [24]:
def net_section_rupture_angles_channels(w:float,t:float,fu:float,fy:float,Ago:float,Anc:float,bs:float,Lc:float)->float:
    """
    """
    gamma_m0=1.1
    gamma_m1=1.25
    beta=1.4-(0.076*(w/t)*(fy/fu)*(bs/Lc))
    T_dn=(((beta*fy/gamma_m0)*Ago)+((0.9*fu/gamma_m1)*Anc))*10**-3
    return round(T_dn,2)

In [14]:
net_section_rupture_angles_channels(60,8,410,250,448,512,100,120)

264.2

In [25]:
def block_shear_failure(fu:float,fy:float,Avg:float,Atn:float,Avn:float,Atg:float)->float:
    """
    """
    gamma_m0=1.1
    gamma_m1=1.25
    T_db1 = ((fy * Avg) / (math.sqrt(3) * gamma_m0)+ (0.9 * fu * Atn) / gamma_m1) * 10**-3
    T_db2 = ((0.9 * fu * Avn) / (math.sqrt(3) * gamma_m1)+ (fy * Atg) / gamma_m0) * 10**-3
    T_db= min(T_db1,T_db2)
    return round(T_db,2)

In [18]:
block_shear_failure(410,250,1600,190,1050,300)

247.14

In [29]:
def design_tensile_strength(fy:float,Ag:float,fu:float,Anet:float,Avg:float,Atn:float,Avn:float,Atg:float)->float:
    """
    Returns the design tensile strength of the member
    """

    T_dg=gross_section_yielding(fy,Ag)
    T_dn=net_section_rupture_plates(fu,Anet)
    T_db=block_shear_failure(fu,fy,Avg,Atn,Avn,Atg)
    T_d=min(T_dg,T_dn,T_db)
    return round(T_d,2)
    
    

In [30]:
design_tensile_strength(250,1500,410,1060,1800,390,1300,500)

312.91

In [10]:
import pandas as pd

df=pd.read_csv('rolled_steel_channels.csv')
df=df.set_index('Section')
rolled_steel_channels=df.copy()

In [11]:
rolled_steel_channels

,W_kg,W_N,Area,h,b,tf,tw,Cyy,Ixx,Iyy,...,r1,r2,D,h1,h2,b1/2,C,g*,g1_min,max_flange_rivet
Section,,,,,,,,,,,,,,,,,,,,,
ISJC 100,5.8,56.9,7.41,100,45,5.1,3.0,1.40,123.8,14.9,...,6.0,2.0,91.5,77.0,11.5,21.0,4.5,25,50.0,12
ISJC 125,7.9,77.5,10.07,125,50,6.6,3.0,1.64,270.0,25.7,...,6.0,2.5,91.5,98.9,13.1,23.5,4.5,28,50.0,16
ISJC 150,9.9,97.1,12.65,150,55,6.9,3.6,1.66,471.1,37.9,...,7.0,3.0,91.5,121.2,14.4,25.7,5.1,30,50.0,20
ISJC 175,11.2,109.9,14.24,175,60,6.9,3.6,1.75,719.9,50.5,...,7.0,3.0,91.5,146.1,14.5,28.2,5.1,35,50.0,20
ISJC 200,13.9,136.4,17.77,200,70,7.1,4.1,1.97,1161.2,84.2,...,8.0,3.5,91.5,158.5,15.8,33.0,5.6,40,50.0,22
ISLC 75,5.7,55.9,7.26,75,40,6.0,3.7,1.35,66.1,11.5,...,5.0,2.0,91.5,50.4,12.3,18.2,5.2,21,NaN,12
ISLC 100,7.9,77.5,10.02,100,50,6.4,4.0,1.62,164.7,24.8,...,6.0,2.0,91.5,74.3,12.8,23.0,5.5,28,50.0,16
ISLC 125,10.7,105.0,13.67,125,65,6.6,4.4,2.04,356.8,57.2,...,7.0,2.5,91.5,95.6,14.2,30.3,5.9,35,50.0,22
ISLC 150,14.4,141.3,18.36,150,75,7.8,4.8,2.38,697.2,103.2,...,8.0,3.5,91.5,117.6,16.5,35.1,6.3,40,50.0,25


In [22]:
def rolled_steel_channel(channel:str,W:float)->float:
    """
    """
    df=pd.read_csv('rolled_steel_channels.csv')
    df=df.set_index('Section')
    rolled_steel_channel=df.copy()
    condition = ((rolled_steel_channel.index == channel) & (rolled_steel_channel["W_N"] == W))
    steel_channel = rolled_steel_channel.loc[condition].iloc[0]
    

    Area = steel_channel["Area"]
    h = steel_channel["h"]
    b = steel_channel["b"]
    tf = steel_channel["tf"]
    tw = steel_channel["tw"]
    rxx = steel_channel["rxx"]
    ryy = steel_channel["ryy"]

    return Area, h, b, tf,tw, rxx, ryy

In [23]:
rolled_steel_channel('ISJC 100',56.9)

(7.41, 100.0, 45.0, 5.1, 3.0, 4.09, 1.42)

In [31]:
def rolled_steel_equal_angle(equal_angle:str,W:float)->float:
    """
    """
    df=pd.read_csv('rolled_steel_equal_angles.csv')
    df=df.set_index('Section')
    rolled_steel_equal_angle=df.copy()
    condition = ((rolled_steel_equal_angle.index == equal_angle) & (rolled_steel_equal_angle["W_N"] == W))
    steel_equal_angle = rolled_steel_equal_angle.loc[condition].iloc[0]
    

    Area = steel_equal_angle["Area"]
    t = steel_equal_angle["t"]
    Ixx = steel_equal_angle["Ixx_Iyy"]
    rxx = steel_equal_angle["rxx_ryy"]
    

    return Area, t,Ixx,rxx

In [43]:
rolled_steel_equal_angle("ISA 2020",8.8)

(1.12, 3.0, 0.4, 0.58)

In [41]:
Area,t,Ixx,rxx=rolled_steel_equal_angle("ISA 2020",8.8)

In [42]:
Area,t,Ixx,rxx

(1.12, 3.0, 0.4, 0.58)

In [26]:
df=pd.read_csv('rolled_steel_equal_angles.csv')
df=df.set_index('Section')
rolled_steel_equal_angle=df.copy()

In [27]:
rolled_steel_equal_angle

,A,B,t,Area,W_kg_m,W_N_m,Cxx_Cyy,exx_eyy,Ixx_Iyy,Iuu,Ivv,rxx_ryy,ruu,rvv,Zxx_Zyy,r1,r2,Ixy
Section,,,,,,,,,,,,,,,,,,
ISA 2020,20,20,3,1.12,0.9,8.8,0.59,1.41,0.4,0.6,0.2,0.58,0.73,0.37,0.3,4.0,2.5,0.2
ISA 2020,20,20,4,1.45,1.1,10.8,0.63,1.37,0.5,0.8,0.2,0.58,0.72,0.37,0.4,4.0,2.5,0.3
ISA 2525,25,25,3,1.41,1.1,10.8,0.71,1.79,0.8,1.2,0.3,0.73,0.93,0.47,0.4,4.5,3.0,0.4
ISA 2525,25,25,4,1.84,1.4,13.7,0.75,1.75,1.0,1.6,0.4,0.73,0.91,0.47,0.6,4.5,3.0,0.6
ISA 2525,25,25,5,2.25,1.8,17.7,0.79,1.71,1.2,1.8,0.5,0.72,0.91,0.47,0.7,4.5,3.0,0.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ISA 150150,150,150,18,50.79,39.9,391.4,4.38,10.62,1048.9,1668.2,429.5,4.54,5.73,2.91,98.7,12.0,8.0,616.0
ISA 200200,200,200,12,46.61,36.6,359.0,5.36,14.64,1788.9,2862.0,715.9,6.20,7.84,3.92,122.2,15.0,10.0,1058.9
ISA 200200,200,200,15,57.80,45.4,445.4,5.49,14.51,2197.7,3511.8,883.7,6.17,7.79,3.91,151.4,15.0,10.0,1301.2


In [36]:
def rolled_steel_unequal_angle(unequal_angle:str,W:float)->float:
    """
    """
    df=pd.read_csv('rolled_steel_unequal_angles.csv')
    df=df.set_index('Section')
    rolled_steel_unequal_angle=df.copy()
    condition = ((rolled_steel_unequal_angle.index == unequal_angle) & (rolled_steel_unequal_angle["W_N"] == W))
    steel_unequal_angle = rolled_steel_unequal_angle.loc[condition].iloc[0]
    

    Area = steel_unequal_angle["Area"]
    t = steel_unequal_angle["t"]
    Ixx = steel_unequal_angle["Ixx"]
    Iyy = steel_unequal_angle["Iyy"]
    rxx = steel_unequal_angle["rxx"]
    ryy = steel_unequal_angle["ryy"]
    

    return Area, t,Ixx,Iyy,rxx,ryy

In [37]:
rolled_steel_unequal_angle("ISA 3020",10.8)

(1.41, 3.0, 1.2, 0.4, 0.92, 0.54)

In [33]:
df=pd.read_csv('rolled_steel_unequal_angles.csv')
df=df.set_index('Section')
rolled_steel_unequal_angle=df.copy()

In [34]:
rolled_steel_unequal_angle

,A,B,t,Area,W_kg,W_N,Cxx,Cyy,exx,eyy,...,rxx,ryy,ruu,rvv,Zxx,Zyy,tan_alpha,r1,r2,Ixy
Section,,,,,,,,,,,,,,,,,,,,,
ISA 3020,30,20,3,1.41,1.1,10.8,0.98,0.49,2.02,1.51,...,0.92,0.54,0.99,0.41,0.6,0.3,0.43,4.5,3.0,0.4
ISA 3020,30,20,4,1.84,1.4,13.7,1.02,0.53,1.98,1.47,...,0.92,0.54,0.98,0.41,0.8,0.4,0.42,4.5,3.0,0.5
ISA 3020,30,20,5,2.25,1.8,17.7,1.06,0.57,1.94,1.43,...,0.91,0.53,0.97,0.41,1.0,0.4,0.41,4.5,3.0,0.6
ISA 4025,40,25,3,1.88,1.5,14.7,1.30,0.57,2.70,1.93,...,1.25,0.68,1.33,0.52,1.1,0.5,0.38,5.0,3.0,0.9
ISA 4025,40,25,4,2.46,1.9,18.6,1.35,0.62,2.65,1.88,...,1.25,0.68,1.32,0.52,1.4,0.6,0.38,5.0,3.0,1.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ISA 200100,200,100,15,42.78,33.6,329.6,7.18,2.22,12.82,7.78,...,6.40,2.64,6.59,2.12,138.6,38.3,0.26,12.0,8.0,395.9
ISA 200150,200,150,10,34.00,26.7,261.9,5.99,3.51,14.01,11.49,...,6.37,4.44,7.08,3.31,98.3,53.3,0.58,13.5,9.5,384.1
ISA 200150,200,150,12,40.56,31.8,312.0,6.08,3.60,13.92,11.40,...,6.35,4.42,7.04,3.31,117.4,66.6,0.58,13.5,9.5,509.1


In [39]:
def rolled_steel_tee_bar(tee_bar:str,W:float)->float:
    """
    """
    df=pd.read_csv('rolled_steel_tee_bars.csv')
    df=df.set_index('Section')
    rolled_steel_tee_bar=df.copy()
    condition = ((rolled_steel_tee_bar.index == tee_bar) & (rolled_steel_tee_bar["W_N"] == W))
    steel_tee_bar = rolled_steel_tee_bar.loc[condition].iloc[0]
    

    Area = steel_tee_bar["Area"]
    h = steel_tee_bar["h"]
    b = steel_tee_bar["b"]
    tf = steel_tee_bar["tf"]
    tw = steel_tee_bar["tw"]
    Ixx = steel_tee_bar["Ixx"]
    Iyy = steel_tee_bar["Iyy"]
    rxx = steel_tee_bar["rxx"]
    ryy = steel_tee_bar["ryy"]
    

    return Area, h,b,tf,tw,Ixx,Iyy,rxx,ryy

In [40]:
rolled_steel_tee_bar("ISNT 20",8.8)

(1.13, 20.0, 20.0, 3.0, 3.0, 0.4, 0.2, 0.59, 0.39)